In [ ]:
import numpy as np
import pandas as pd
import re
import torch
import pickle
from collections import defaultdict, Counter

from transformers import AutoTokenizer, AutoModel

from sklearn.cluster import KMeans, DBSCAN
from scipy.cluster.hierarchy import linkage, fcluster, dendrogram
from sklearn.metrics import (
    silhouette_score, calinski_harabasz_score, davies_bouldin_score,
    adjusted_rand_score, normalized_mutual_info_score, homogeneity_score,
    completeness_score, v_measure_score
)
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
from sklearn.preprocessing import LabelEncoder, StandardScaler

import matplotlib.pyplot as plt
import seaborn as sns

import warnings
warnings.filterwarnings('ignore')

from transformers import pipeline

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Getting embeddings

### Data loading

In [ ]:
def check_conllu(file_path):
    sentences = []
    current_sent = []
    sent_id = None
    sent_text = None

    with open(file_path, 'r', encoding='utf-8') as text:
        for line in text:
            line = line.rstrip()

            if not line:
                # Пустая строка - конец предложения
                if current_sent and sent_id and sent_text:
                    sentences.append({
                        'sent_id': sent_id,
                        'text': sent_text,
                        'tokens': current_sent.copy()
                    })
                    current_sent = []
                    sent_id = None
                    sent_text = None
                continue

            # Обработка комментариев (метаданные)
            if line.startswith('#'):
                if 'sent_id' in line:
                    sent_id = line.split('=', 1)[1].strip()
                elif 'text' in line:
                    sent_text = line.split('=', 1)[1].strip()
                continue

            parts = line.split('\t')
            if len(parts) != 12:
              print(sent_id)

file_path = '/content/drive/MyDrive/Master_thesis/train.conllu'
check_conllu(file_path)

In [ ]:
def read_conll_format(file_path):
    sentences = []
    current_sent = []
    sent_id = None
    sent_text = None

    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.rstrip()

            if not line:
                # Конец предложения
                if current_sent and sent_id is not None:
                    sentences.append({
                        'id': sent_id,
                        'text': sent_text or '',
                        'tokens': current_sent.copy()
                    })
                current_sent = []
                sent_id = None
                sent_text = None
                continue

            if line.startswith('#'):
                if line.startswith('# sent_id ='):
                    sent_id = line.split('=', 1)[1].strip()
                elif line.startswith('# text ='):
                    sent_text = line.split('=', 1)[1].strip()
                continue

            parts = line.split('\t')

            token = {
                'id': parts[0],
                'form': parts[1],
                'lemma': parts[2],
                'upos': parts[3],
                'xpos': parts[4],
                'feats': parts[5],
                'head': parts[6],
                'deprel': parts[7],
                'deps': parts[8],
                'misc': parts[9],
                'sem_role': parts[10],
                'sem_class': parts[11]
            }
            current_sent.append(token)
    return sentences

In [ ]:
conll_data = read_conll_format('/content/drive/MyDrive/Master_thesis/train.conllu')

In [ ]:
conll_data[0]

{'id': '8193',
 'text': 'Среди них - статуя Христа - Спасителя и гора Сахарная голова в Рио-де- Жанейро, площадь Согласия в Париже, Бранденбургские ворота в Берлине.',
 'tokens': [{'id': '1',
   'form': 'Среди',
   'lemma': 'среди',
   'upos': 'ADP',
   'xpos': 'Preposition',
   'feats': '_',
   'head': '2',
   'deprel': 'case',
   'deps': '2:case',
   'misc': '_',
   'sem_role': '_',
   'sem_class': 'PREPOSITION'},
  {'id': '2',
   'form': 'них',
   'lemma': 'он',
   'upos': 'PRON',
   'xpos': 'Pronoun',
   'feats': 'Case=Gen|Number=Plur|Person=3',
   'head': '0',
   'deprel': 'root',
   'deps': '2.1:obl',
   'misc': '_',
   'sem_role': 'SetEnvironment',
   'sem_class': 'ENTITY_OR_SITUATION_PRONOUN'},
  {'id': '2.1',
   'form': '#NULL',
   'lemma': 'быть',
   'upos': 'VERB',
   'xpos': 'Verb',
   'feats': '_',
   'head': '_',
   'deprel': '_',
   'deps': '0:root',
   'misc': 'ellipsis',
   'sem_role': 'Predicate',
   'sem_class': 'EXISTENCE_AND_POSSESSION'},
  {'id': '3',
   'form': '

In [ ]:
def analyze_corpus_with_context(conll_data):
    print("АНАЛИЗ КОРПУСА CoBaLD Rus")
    print()
    print(f"Всего предложений: {len(conll_data)}")

    total_tokens = 0
    unique_forms = set()
    context_unique_tokens = set()
    sem_classes = {}
    sem_roles = {}
    upos_tags = {}
    form_to_sem_classes = {}

    for sent_idx, sent in enumerate(conll_data):
        for token_idx, token in enumerate(sent['tokens']):
            total_tokens += 1

            unique_forms.add(token['form'])
            context_token = (token['form'], token['sem_class'], token['sem_role'])
            context_unique_tokens.add(context_token)

            if token['sem_class'] != '_':
                sem_classes[token['sem_class']] = sem_classes.get(token['sem_class'], 0) + 1

            if token['sem_role'] != '_':
                sem_roles[token['sem_role']] = sem_roles.get(token['sem_role'], 0) + 1

            upos_tags[token['upos']] = upos_tags.get(token['upos'], 0) + 1

            form = token['form']
            sem_class = token['sem_class']

            if form not in form_to_sem_classes:
                form_to_sem_classes[form] = set()
            if sem_class != '_':
                form_to_sem_classes[form].add(sem_class)

    print(f"Всего токенов (вхождений): {total_tokens}")
    print(f"Уникальных форм (типов): {len(unique_forms)}")
    print(f"Уникальных контекстных токенов: {len(context_unique_tokens)}")

    print(f"\nСЕМАНТИЧЕСКАЯ РАЗМЕТКА:")
    print(f"  Уникальных семантических классов: {len(sem_classes)}")
    print(f"  Уникальных семантических ролей: {len(sem_roles)}")
    print(f"  Уникальных частей речи: {len(upos_tags)}")
    print(upos_tags)
    print()

    print(f"\nАНАЛИЗ МНОГОЗНАЧНОСТИ:")
    ambiguous_words = []
    for form, classes in form_to_sem_classes.items():
        if len(classes) > 1:
            ambiguous_words.append((form, len(classes), classes))

    ambiguous_words.sort(key=lambda x: x[1], reverse=True)

    print(f"  Слов с разными семантическими классами: {len(ambiguous_words)}")

    if ambiguous_words:
        print(f"\n  Топ-10 самых многозначных слов:")
        for form, count, classes in ambiguous_words[:10]:
            print(f"    '{form}': {count} классов - {', '.join(sorted(list(classes))[:3])}")


result = analyze_corpus_with_context(conll_data)

АНАЛИЗ КОРПУСА CoBaLD Rus

Всего предложений: 27520
Всего токенов (вхождений): 357589
Уникальных форм (типов): 58143
Уникальных контекстных токенов: 91475

СЕМАНТИЧЕСКАЯ РАЗМЕТКА:
  Уникальных семантических классов: 565
  Уникальных семантических ролей: 133
  Уникальных частей речи: 21
{'ADP': 37333, 'PRON': 12697, 'VERB': 46626, 'PUNCT': 60655, 'NOUN': 97053, 'PROPN': 25142, 'CCONJ': 6452, 'ADJ': 27042, 'AUX': 2806, 'PART': 5293, 'ADV': 13348, 'DET': 6425, 'SCONJ': 4435, 'NUM': 9689, 'SYM': 640, 'X': 1521, '_': 398, 'INTJ': 26, 'W': 5, 'Invariable': 2, 'Prefixoid': 1}


АНАЛИЗ МНОГОЗНАЧНОСТИ:
  Слов с разными семантическими классами: 4846

  Топ-10 самых многозначных слов:
    '#NULL': 114 классов - ABILITY_OF_BEING, ACT, ADMINISTRATIVE_REGION
    'его': 69 классов - ACTIVITY, AGGRESSIVE_ACTIONS, APPARATUS
    'которые': 66 классов - ACT, ADMINISTRATIVE_REGION, AGGREGATE
    'который': 59 классов - ACT, AGGREGATE, APPARATUS
    'которой': 50 классов - ADMINISTRATIVE_REGION, AGGRESSIVE

### utils

In [ ]:
def get_all_embeddings(conll_data, model, tokenizer, device='cuda' if torch.cuda.is_available() else 'cpu'):
    embeddings = {}
    model.to(device)
    model.eval()

    for idx, sent_data in enumerate(conll_data):
        if idx % 500 == 0:
          print(idx)
        sent_id = sent_data['id']
        tokens = sent_data['tokens']

        word_forms = [token['form'] for token in tokens]

        if not word_forms:
            continue

        inputs = tokenizer(
            word_forms,
            is_split_into_words=True,
            return_tensors="pt",
            truncation=True,
            padding=True,
            return_offsets_mapping=False
        )
        batch_encoding = inputs
        inputs_to_model = {k: v.to(device) for k, v in inputs.items()}

        # Получаем эмбеддинги токенов
        with torch.no_grad():
            model_output = model(**inputs_to_model)
        token_embeddings = model_output.last_hidden_state[0].float().detach().cpu().numpy()

        # Получаем mapping токенов к словам (word_ids) ИЗ СОХРАНЕННОГО ОБЪЕКТА
        # word_ids содержит None для специальных токенов ([CLS], [SEP], [PAD])
        # и индексы слов для обычных токенов
        word_ids = batch_encoding.word_ids(batch_index=0)

        word_embeddings_dict = {}
        for token_idx, (embedding, word_id) in enumerate(zip(token_embeddings, word_ids)):
            if word_id is not None:  # Пропускаем специальные токены (word_ids() is None)
                if word_id not in word_embeddings_dict:
                    word_embeddings_dict[word_id] = []
                word_embeddings_dict[word_id].append(embedding)

        for token_idx, token in enumerate(tokens):

            if token_idx in word_embeddings_dict and word_embeddings_dict[token_idx]:
                subword_embeddings = word_embeddings_dict[token_idx]

                # Усредняем эмбеддинги подслов для получения эмбеддинга слова
                avg_embedding = np.mean(subword_embeddings, axis=0)

                key = (sent_id, token_idx)
                embeddings[key] = {
                    'embedding': avg_embedding,
                    'form': token['form'],
                    'lemma': token['lemma'],
                    'upos': token['upos'],
                    'xpos': token['xpos'],
                    'feats': token['feats'],
                    'head': token['head'],
                    'deprel': token['deprel'],
                    'sem_role': token['sem_role'],
                    'sem_class': token['sem_class'],
                    'context': sent_data.get('text', ''),
                    'position_in_sentence': token_idx
                }
            else:
                print(f"ВНИМАНИЕ: слово '{token['form']}' (индекс {token_idx}) в предложении {sent_id} не получило эмбеддинг")

    return embeddings

### Models

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
# import embed_anything_gpu
# from embed_anything_gpu import EmbeddingModel, TextEmbedConfig
# model = EmbeddingModel.from_pretrained_hf(
#     WhichModel.Bert,
#     model_id="sentence-transformers/all-MiniLM-L6-v2"
# )
# config = TextEmbedConfig(chunk_size=200, batch_size=32)
# data = embed_anything.embed_file("text.txt", embeder=model, config=config)

In [ ]:
tokenizer_frida = AutoTokenizer.from_pretrained('ai-forever/FRIDA')
model_frida = AutoModel.from_pretrained('ai-forever/FRIDA').to(device)

config.json:   0%|          | 0.00/823 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/958 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.29G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/219 [00:00<?, ?it/s]

T5Model LOAD REPORT from: ai-forever/FRIDA
Key                                                                  | Status  | 
---------------------------------------------------------------------+---------+-
decoder.block.{0...23}.layer.0.SelfAttention.v.weight                | MISSING | 
decoder.block.{0...23}.layer.2.DenseReluDense.wi_0.weight            | MISSING | 
decoder.block.{0...23}.layer.1.EncDecAttention.k.weight              | MISSING | 
decoder.block.{0...23}.layer.0.SelfAttention.o.weight                | MISSING | 
decoder.block.{0...23}.layer.0.SelfAttention.q.weight                | MISSING | 
decoder.block.{0...23}.layer.2.DenseReluDense.wi_1.weight            | MISSING | 
decoder.block.{0...23}.layer.1.EncDecAttention.o.weight              | MISSING | 
decoder.block.{0...23}.layer.{0, 1, 2}.layer_norm.weight             | MISSING | 
decoder.block.{0...23}.layer.0.SelfAttention.k.weight                | MISSING | 
decoder.block.{0...23}.layer.1.EncDecAttention.q.weight

In [ ]:
tokenizer_e5 = AutoTokenizer.from_pretrained('intfloat/multilingual-e5-large')
model_e5 = AutoModel.from_pretrained('intfloat/multilingual-e5-large').to(device)

config.json:   0%|          | 0.00/690 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/418 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.24G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: intfloat/multilingual-e5-large
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
tokenizer_qwen2b = AutoTokenizer.from_pretrained('Qwen/Qwen3-VL-2B-Instruct')
model_qwen2b = AutoModel.from_pretrained('Qwen/Qwen3-VL-2B-Instruct').to(device)

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/4.26G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/625 [00:00<?, ?it/s]

In [ ]:
tokenizer_sbert = AutoTokenizer.from_pretrained('ai-forever/sbert_large_nlu_ru')
model_sbert = AutoModel.from_pretrained('ai-forever/sbert_large_nlu_ru').to(device)

config.json:   0%|          | 0.00/863 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.71G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

In [ ]:
tokenizer_qwen_small = AutoTokenizer.from_pretrained('Qwen/Qwen3-1.7B')
model_qwen_small = AutoModel.from_pretrained('Qwen/Qwen3-1.7B').to(device)

config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

Qwen3Model LOAD REPORT from: Qwen/Qwen3-1.7B
Key            | Status     |  | 
---------------+------------+--+-
lm_head.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
# names = ['frida', 'e5', 'qwen2b', 'sbert', 'qwen_small']
# emb = []
# name_to_mod = {
#     'frida': [tokenizer_frida, model_frida],
#     'e5': [tokenizer_e5, model_e5],
#     'qwen2b': [tokenizer_qwen2b, model_qwen2b], ,
#     'sbert': [tokenizer_sbert, model_sbert], ,
#     'qwen_small': [tokenizer_qwen_small, model_qwen_small]
# }

In [ ]:
tokenizer_qwen_small = AutoTokenizer.from_pretrained('Xenova/paraphrase-multilingual-MiniLM-L12-v2')
model_qwen_small = AutoModel.from_pretrained('Xenova/paraphrase-multilingual-MiniLM-L12-v2').to(device)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained('BAAI/bge-large-en-v1.5')
model = AutoModel.from_pretrained('BAAI/bge-large-en-v1.5').to(device)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained('FacebookAI/xlm-roberta-base')
model = AutoModel.from_pretrained('FacebookAI/xlm-roberta-base').to(device)

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.12G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: FacebookAI/xlm-roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
tokenizer = AutoTokenizer.from_pretrained('51la5/roberta-large-NER')
model = AutoModel.from_pretrained('51la5/roberta-large-NER').to(device)

config.json:   0%|          | 0.00/852 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/2.24G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.24G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: 51la5/roberta-large-NER
Key               | Status     |  | 
------------------+------------+--+-
classifier.weight | UNEXPECTED |  | 
classifier.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
tokenizer = AutoTokenizer.from_pretrained('Babelscape/wikineural-multilingual-ner')
model = AutoModel.from_pretrained('Babelscape/wikineural-multilingual-ner').to(device)

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/333 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/709M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertModel LOAD REPORT from: Babelscape/wikineural-multilingual-ner
Key                          | Status     | 
-----------------------------+------------+-
bert.embeddings.position_ids | UNEXPECTED | 
classifier.weight            | UNEXPECTED | 
classifier.bias              | UNEXPECTED | 
pooler.dense.bias            | MISSING    | 
pooler.dense.weight          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
tokenizer = AutoTokenizer.from_pretrained('FacebookAI/xlm-roberta-large-finetuned-conll03-english')
model = AutoModel.from_pretrained('FacebookAI/xlm-roberta-large-finetuned-conll03-english').to(device)

config.json:   0%|          | 0.00/852 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/2.24G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: FacebookAI/xlm-roberta-large-finetuned-conll03-english
Key               | Status     |  | 
------------------+------------+--+-
classifier.weight | UNEXPECTED |  | 
classifier.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


## Embeddings

In [ ]:
# get_all_embeddings([conll_data[0]], model_frida, tokenizer_frida)

In [ ]:
get_all_embeddings([conll_data[0]], model, tokenizer)

0


{('8193',
  0): {'embedding': array([1.0892274 , 1.3997791 , 0.18246678, ..., 0.5870178 , 0.53950685,
         1.3710288 ], dtype=float32), 'form': 'Среди', 'lemma': 'среди', 'upos': 'ADP', 'xpos': 'Preposition', 'feats': '_', 'head': '2', 'deprel': 'case', 'sem_role': '_', 'sem_class': 'PREPOSITION', 'context': 'Среди них - статуя Христа - Спасителя и гора Сахарная голова в Рио-де- Жанейро, площадь Согласия в Париже, Бранденбургские ворота в Берлине.', 'position_in_sentence': 0},
 ('8193',
  1): {'embedding': array([1.1765015 , 1.3544111 , 0.07098383, ..., 1.0985895 , 0.2571051 ,
         1.2973242 ], dtype=float32), 'form': 'них', 'lemma': 'он', 'upos': 'PRON', 'xpos': 'Pronoun', 'feats': 'Case=Gen|Number=Plur|Person=3', 'head': '0', 'deprel': 'root', 'sem_role': 'SetEnvironment', 'sem_class': 'ENTITY_OR_SITUATION_PRONOUN', 'context': 'Среди них - статуя Христа - Спасителя и гора Сахарная голова в Рио-де- Жанейро, площадь Согласия в Париже, Бранденбургские ворота в Берлине.', 'positi

In [ ]:
all_embeddings = get_all_embeddings(conll_data, model, tokenizer)

0
500
1000
1500
2000
2500
3000
3500
4000
4500
5000
5500
6000
6500
7000
7500
8000
8500
9000
9500
10000
10500
11000
11500
12000
12500
13000
13500
14000
14500
15000
15500
16000
16500
17000
17500
18000
18500
19000
19500
20000
20500
21000
21500
22000
22500
23000
23500
24000
24500
25000
25500
26000
26500
27000
27500


In [ ]:
print(f"Всего получено эмбеддингов: {len(all_embeddings)}")

import pickle

with open('/content/drive/MyDrive/Master_thesis/robertanew_embeddings.pkl', 'wb') as f:
    pickle.dump(all_embeddings, f)

Всего получено эмбеддингов: 357589


In [ ]:
# Пример: находим все вхождения слова "которых" (если есть)
entries = []
for key, data in all_embeddings.items():
    if data['form'].lower() == 'которых':
        entries.append((key, data))

print(f"\nНайдено вхождений слова 'которых': {len(entries)}")
if len(entries) >= 3:
    from sklearn.metrics.pairwise import cosine_similarity

    emb1 = entries[0][1]['embedding'].reshape(1, -1)
    emb2 = entries[1][1]['embedding'].reshape(1, -1)

    similarity = cosine_similarity(emb1, emb2)[0][0]
    print(f"Косинусное сходство между разными вхождениями 'которых': {similarity:.4f}")

    print(f"\nКонтекст 1 (предложение {entries[0][0][0]}, токен {entries[0][0][1]}):")
    print(f"  {entries[0][1]['context']}")
    print(f"\nКонтекст 2 (предложение {entries[1][0][0]}, токен {entries[1][0][1]}):")
    print(f"  {entries[1][1]['context']}")


Найдено вхождений слова 'которых': 103
Косинусное сходство между разными вхождениями 'которых': 0.9779

Контекст 1 (предложение 6726, токен 30):
  Адвокат указал, что под руководством Козлова в 2005-2006 годах были отозваны лицензии у порядка 300 кредитных учреждений, "в том числе очень крупных банков, деятельность которых была невозможна без поддержки".

Контекст 2 (предложение 10917, токен 4):
  Передислокация ЗРК, разработка которых при участии США обошлась Японии в 8 млрд долл., полностью завершится 31 марта, сообщает ИТАР-ТАСС.


## Clustering

### data loading

In [ ]:
import pickle

with open('/content/drive/MyDrive/Master_thesis/word_embeddings.pkl', 'rb') as f:
    all_embeddings = pickle.load(f)

In [ ]:
full = all_embeddings.copy()

In [ ]:
all_embeddings = dict(list(all_embeddings.items())[:10])

In [ ]:
all_embeddings

{('8193',
  0): {'embedding': array([ 0.51826984, -0.16849124, -0.7175239 , ..., -0.53683287,
          0.7626474 , -0.05710888], dtype=float32), 'form': 'Среди', 'lemma': 'среди', 'upos': 'ADP', 'xpos': 'Preposition', 'feats': '_', 'head': '2', 'deprel': 'case', 'sem_role': '_', 'sem_class': 'PREPOSITION', 'context': 'Среди них - статуя Христа - Спасителя и гора Сахарная голова в Рио-де- Жанейро, площадь Согласия в Париже, Бранденбургские ворота в Берлине.', 'position_in_sentence': 0},
 ('8193',
  1): {'embedding': array([ 0.4911438 , -0.27209017, -0.2505134 , ..., -0.45973575,
          0.0958458 , -0.20058353], dtype=float32), 'form': 'них', 'lemma': 'он', 'upos': 'PRON', 'xpos': 'Pronoun', 'feats': 'Case=Gen|Number=Plur|Person=3', 'head': '0', 'deprel': 'root', 'sem_role': 'SetEnvironment', 'sem_class': 'ENTITY_OR_SITUATION_PRONOUN', 'context': 'Среди них - статуя Христа - Спасителя и гора Сахарная голова в Рио-де- Жанейро, площадь Согласия в Париже, Бранденбургские ворота в Берлин

In [ ]:
all_embeddings = full.copy()

In [ ]:
cleaned_embeddings = {}

null_count = 0
total_count = len(all_embeddings)

for key, data in all_embeddings.items():
    if isinstance(data, dict):
        form = data.get('form', '')

        if form == '#NULL':
            null_count += 1
            continue

        if 'embedding' not in data:
            print(f"Внимание: у ключа {key} нет эмбеддинга, пропускаем")
            continue

        cleaned_embeddings[key] = data
    else:
        print(f"Внимание: неправильный формат данных для ключа {key}")

print(f"Статистика очистки:")
print(f"  Исходное количество токенов: {total_count}")
print(f"  Удалено #NULL токенов: {null_count}")
print(f"  Оставлено токенов: {len(cleaned_embeddings)}")
print(f"  Процент удаленных: {null_count/total_count*100:.1f}%")

all_embeddings = cleaned_embeddings.copy()

Статистика очистки:
  Исходное количество токенов: 357589
  Удалено #NULL токенов: 5092
  Оставлено токенов: 352497
  Процент удаленных: 1.4%


### utils without scaling

In [ ]:
def prepare_all_tokens_with_threshold(all_embeddings_dict, min_freq_threshold=3):
    all_tokens_info = []
    all_embeddings = []
    all_sem_classes = []

    for key, data in all_embeddings_dict.items():
        sent_id, token_idx = key

        sem_class = data.get('sem_class', '').strip()
        if not sem_class or sem_class == '_':
            continue
        all_tokens_info.append({
            'sentence_id': sent_id,
            'token_idx': token_idx,
            'form': data['form'],
            'lemma': data.get('lemma', ''),
            'sem_class': sem_class,
            'sem_role': data.get('sem_role', '').strip(),
            'upos': data.get('upos', ''),
            # 'context': data.get('context', '')[:100]
        })

        all_embeddings.append(data['embedding'])
        all_sem_classes.append(sem_class)

    X = np.array(all_embeddings)
    true_labels = np.array(all_sem_classes)

    class_counts = Counter(true_labels)

    print(f"Всего токенов: {len(all_tokens_info)}")
    print(f"Уникальных семантических классов: {len(class_counts)}")
    print(f"Размерность эмбеддингов: {X.shape}")

    filtered_indices = []
    filtered_classes = []
    filtered_tokens_info = []

    for i, sem_class in enumerate(true_labels):
        if class_counts[sem_class] >= min_freq_threshold:
            filtered_indices.append(i)
            filtered_classes.append(sem_class)
            filtered_tokens_info.append(all_tokens_info[i])

    X_filtered = X[filtered_indices]
    true_labels_filtered = np.array(filtered_classes)

    return X_filtered, true_labels_filtered, filtered_tokens_info, filtered_indices, class_counts

def analyze_frequency_distribution(all_embeddings_dict, thresholds=[250, 500, 1000, 1250, 1500, 2000, 3000, 4000, 5000]):
    all_sem_classes = []
    for data in all_embeddings_dict.values():
        sem_class = data.get('sem_class', '').strip()
        if sem_class and sem_class != '_':
            all_sem_classes.append(sem_class)

    class_counts = Counter(all_sem_classes)

    results = []
    for threshold in thresholds:
        classes_above = sum(1 for count in class_counts.values() if count >= threshold)
        tokens_above = sum(count for count in class_counts.values() if count >= threshold)

        results.append({
            'threshold': threshold,
            'classes': classes_above,
            'tokens': tokens_above,
            'tokens_percent': tokens_above / len(all_sem_classes) * 100 if all_sem_classes else 0
        })

    df_results = pd.DataFrame(results)

    fig, axes = plt.subplots(2, 2, figsize=(15, 12))

    # 1. Количество классов
    axes[0, 0].plot(df_results['threshold'], df_results['classes'], 'bo-', linewidth=2)
    axes[0, 0].set_xlabel('Порог частоты')
    axes[0, 0].set_ylabel('Количество классов')
    axes[0, 0].set_title('Классы с частотой ≥ порога')
    axes[0, 0].grid(True, alpha=0.3)

    # 2. Количество токенов
    axes[0, 1].plot(df_results['threshold'], df_results['tokens'], 'ro-', linewidth=2)
    axes[0, 1].set_xlabel('Порог частоты')
    axes[0, 1].set_ylabel('Количество токенов')
    axes[0, 1].set_title('Токены в классах с частотой ≥ порога')
    axes[0, 1].grid(True, alpha=0.3)

    # 3. Процент токенов
    axes[1, 0].plot(df_results['threshold'], df_results['tokens_percent'], 'go-', linewidth=2)
    axes[1, 0].set_xlabel('Порог частоты')
    axes[1, 0].set_ylabel('Процент токенов (%)')
    axes[1, 0].set_title('Доля токенов в классах с частотой ≥ порога')
    axes[1, 0].grid(True, alpha=0.3)
    axes[1, 0].set_ylim([0, 100])

    # 4. Распределение частот (log scale)
    frequencies = sorted(class_counts.values(), reverse=True)
    axes[1, 1].plot(range(1, len(frequencies) + 1), frequencies, 'mo-', linewidth=1, alpha=0.7)
    axes[1, 1].set_xlabel('Ранг класса')
    axes[1, 1].set_ylabel('Частота')
    axes[1, 1].set_title('Распределение Zipf (лог. масштаб)')
    axes[1, 1].grid(True, alpha=0.3)
    axes[1, 1].set_yscale('log')

    plt.suptitle('Анализ распределения семантических классов', fontsize=16)
    plt.tight_layout()
    plt.show()

    print("\nТаблица распределения:")
    print(df_results.to_string(index=False))

    return df_results, class_counts

In [ ]:
def perform_kmeans_clustering(X, true_labels, n_clusters=400):
    n_clusters=len(Counter(true_labels))

    if len(X) < n_clusters:
        print(f"Предупреждение: данных ({len(X)}) меньше чем кластеров ({n_clusters})")
        n_clusters = min(len(X) // 2, 50)
        print(f"Используем k={n_clusters}")

    kmeans = KMeans(
        n_clusters=n_clusters,
        random_state=42,
        n_init=10,
        max_iter=500,
        verbose=1
    )

    print("Выполняется кластеризация...")
    kmeans.fit(X)
    cluster_labels = kmeans.labels_

    unique_clusters = np.unique(cluster_labels)
    n_clusters_found = len(unique_clusters)
    cluster_sizes = np.bincount(cluster_labels)

    print(f"\nРезультаты кластеризации:")
    print(f"  Найдено кластеров: {n_clusters_found}")
    print(f"  Размер кластеров:")
    print(f"    Средний: {np.mean(cluster_sizes):.1f}")
    print(f"    Медиана: {np.median(cluster_sizes):.1f}")
    print(f"    Минимум: {np.min(cluster_sizes)}")
    print(f"    Максимум: {np.max(cluster_sizes)}")

    # Метрики кластеризации
    # if n_clusters_found > 1:
    #     try:
    #         silhouette = silhouette_score(X, cluster_labels)
    #         # calinski = calinski_harabasz_score(X, cluster_labels)
    #         # davies = davies_bouldin_score(X, cluster_labels)

    #         print(f"\nМетрики кластеризации:")
    #         print(f"  Silhouette Score: {silhouette:.4f}")
    #         # print(f"  Calinski-Harabasz: {calinski:.2f}")
    #         # print(f"  Davies-Bouldin: {davies:.4f}")
    #     except Exception as e:
    #         print(f"Не удалось вычислить метрики кластеризации: {e}")
    #         silhouette = calinski = davies = None
    # else:
    #     silhouette = calinski = davies = None

    # Метрики соответствия семантическим классам
    label_encoder = LabelEncoder()
    encoded_true_labels = label_encoder.fit_transform(true_labels)

    ari = adjusted_rand_score(encoded_true_labels, cluster_labels)
    nmi = normalized_mutual_info_score(encoded_true_labels, cluster_labels)
    homogeneity = homogeneity_score(encoded_true_labels, cluster_labels)
    completeness = completeness_score(encoded_true_labels, cluster_labels)
    v_measure = v_measure_score(encoded_true_labels, cluster_labels)

    print(f"\nСоответствие семантическим классам:")
    print(f"  Adjusted Rand Index: {ari:.4f}")
    print(f"  Normalized Mutual Info: {nmi:.4f}")
    print(f"  Homogeneity: {homogeneity:.4f}")
    print(f"  Completeness: {completeness:.4f}")
    print(f"  V-measure: {v_measure:.4f}")

    metrics = {
        # 'silhouette': silhouette,
        # 'calinski': calinski,
        # 'davies': davies,
        'ari': ari,
        'nmi': nmi,
        'homogeneity': homogeneity,
        'completeness': completeness,
        'v_measure': v_measure
    }

    return kmeans, cluster_labels, metrics, label_encoder

In [ ]:
def analyze_clusters_content(tokens_info, cluster_labels, true_labels, top_n_clusters=15):
    df = pd.DataFrame(tokens_info)
    df['cluster'] = cluster_labels
    df['sem_class'] = true_labels

    cluster_sizes = df['cluster'].value_counts()
    total_clusters = len(cluster_sizes)

    print(f"Всего кластеров: {total_clusters}")
    print(f"Всего токенов: {len(df)}")
    print(f"Средний размер кластера: {cluster_sizes.mean():.1f}")

    top_clusters = cluster_sizes.head(top_n_clusters)

    print(f"\nТоп-{top_n_clusters} кластеров по размеру:")

    cluster_details = {}
    for cluster_id, size in top_clusters.items():
        cluster_data = df[df['cluster'] == cluster_id]

        # Самые частые семантические классы
        top_classes = cluster_data['sem_class'].value_counts().head(3)

        # Самые частые словоформы
        top_forms = cluster_data['form'].value_counts().head(5)

        # Уникальные СК
        unique_classes = cluster_data['sem_class'].nunique()

        # Чистота (доля основного СК)
        purity = top_classes.iloc[0] / size if len(top_classes) > 0 else 0

        cluster_details[cluster_id] = {
            'size': size,
            'unique_classes': unique_classes,
            'purity': purity,
            'top_classes': dict(top_classes),
            'top_forms': list(top_forms.index)
        }

        print(f"\nКластер {cluster_id} (размер: {size}):")
        print(f"  Уникальных СК: {unique_classes}")
        print(f"  Чистота: {purity:.1%}")
        print(f"  Топ СК:")
        for cls, count in top_classes.items():
            print(f"    {cls}: {count} ({count/size*100:.1f}%)")
        if top_forms.any():
            print(f"  Топ форм: {', '.join(cluster_details[cluster_id]['top_forms'][:5])}")
    purities = [details['purity'] for details in cluster_details.values()]
    if purities:
        print(f"\nСтатистика чистоты (топ-{top_n_clusters}):")
        print(f"  Средняя: {np.mean(purities):.1%}")
        print(f"  Медиана: {np.median(purities):.1%}")
        print(f"  Минимум: {np.min(purities):.1%}")
        print(f"  Максимум: {np.max(purities):.1%}")

    high_purity_clusters = [cid for cid, details in cluster_details.items() if details['purity'] > 0.8]
    print(f"\nКластеры с чистотой > 80%: {len(high_purity_clusters)}")

    return df, cluster_details

In [ ]:
def visualize_clusters_2d(X, cluster_labels, true_labels, max_points=3000):
    X_sample = X
    clusters_sample = cluster_labels
    true_sample = true_labels

    print("Уменьшение размерности...")
    if X_sample.shape[1] > 50:
        pca = PCA(n_components=min(50, X_sample.shape[0]), random_state=42)
        X_reduced = pca.fit_transform(X_sample)
    else:
        X_reduced = X_sample

    perplexity = min(30, len(X_reduced) - 1)
    tsne = TSNE(
        n_components=2,
        random_state=42,
        perplexity=perplexity,
        n_iter=1000,
        learning_rate='auto',
        init='random',
        verbose=0
    )

    print("Выполняется t-SNE...")
    X_tsne = tsne.fit_transform(X_reduced)
    label_encoder = LabelEncoder()
    encoded_true = label_encoder.fit_transform(true_sample)

    fig, axes = plt.subplots(1, 2, figsize=(20, 8))
    scatter1 = axes[0].scatter(
        X_tsne[:, 0], X_tsne[:, 1],
        c=clusters_sample,
        cmap='tab20',
        alpha=0.7,
        s=20,
        edgecolors='w',
        linewidths=0.3
    )
    axes[0].set_title(f'K-means кластеризация\n{len(np.unique(clusters_sample))} кластеров', fontsize=14)
    axes[0].set_xlabel('t-SNE 1', fontsize=12)
    axes[0].set_ylabel('t-SNE 2', fontsize=12)
    plt.colorbar(scatter1, ax=axes[0], label='Кластер')

    n_true_classes = len(np.unique(encoded_true))
    cmap = 'tab20' if n_true_classes <= 20 else 'viridis'

    scatter2 = axes[1].scatter(
        X_tsne[:, 0], X_tsne[:, 1],
        c=encoded_true,
        cmap=cmap,
        alpha=0.7,
        s=20,
        edgecolors='w',
        linewidths=0.3
    )
    axes[1].set_title(f'Семантические классы\n{n_true_classes} классов', fontsize=14)
    axes[1].set_xlabel('t-SNE 1', fontsize=12)
    axes[1].set_ylabel('t-SNE 2', fontsize=12)
    plt.colorbar(scatter2, ax=axes[1], label='Сем. класс (код)')

    plt.suptitle(f'Визуализация {len(X_sample)} токенов', fontsize=16)
    plt.tight_layout()
    plt.show()

    return X_tsne

In [ ]:
def visualize_confusion_matrix_interactive(cluster_labels, true_labels, top_n_classes=20):
    class_counts = Counter(true_labels)
    top_classes = [cls for cls, _ in class_counts.most_common(top_n_classes)]

    mask = np.isin(true_labels, top_classes)
    filtered_true = true_labels[mask]
    filtered_clusters = cluster_labels[mask]

    df_conf = pd.DataFrame({
        'sem_class': filtered_true,
        'cluster': filtered_clusters
    })

    pivot = df_conf.pivot_table(
        index='sem_class',
        columns='cluster',
        aggfunc='size',
        fill_value=0
    )

    pivot = pivot.loc[top_classes]
    pivot_norm = pivot.div(pivot.sum(axis=1), axis=0)

    fig, axes = plt.subplots(1, 2, figsize=(20, 10))
    sns.heatmap(pivot, ax=axes[0], cmap='Blues',
                cbar_kws={'label': 'Количество'})
    axes[0].set_title('Матрица confusion', fontsize=14)
    axes[0].set_xlabel('Кластер', fontsize=12)
    axes[0].set_ylabel('Семантический класс', fontsize=12)

    sns.heatmap(pivot_norm, ax=axes[1], cmap='Blues',
                cbar_kws={'label': 'Доля'}, vmin=0, vmax=1)
    axes[1].set_title('Матрица confusion (нормализованная)', fontsize=14)
    axes[1].set_xlabel('Кластер', fontsize=12)
    axes[1].set_ylabel('Семантический класс', fontsize=12)

    plt.suptitle(f'Соответствие кластеров и семантических классов\n({len(filtered_true)} токенов, {len(top_classes)} классов)',
                 fontsize=16)
    plt.tight_layout()
    plt.show()

    print(f"\nАнализ для топ-{min(10, len(top_classes))} классов:")

    for cls in top_classes[:10]:
        cls_data = df_conf[df_conf['sem_class'] == cls]
        total = len(cls_data)

        if total > 0:
            main_cluster = cls_data['cluster'].mode().iloc[0]
            main_count = (cls_data['cluster'] == main_cluster).sum()
            main_pct = main_count / total * 100

            print(f"\n  {cls}:")
            print(f"    Токенов: {total}")
            print(f"    Основной кластер: {main_cluster} ({main_pct:.1f}%)")
            print(f"    Уникальных кластеров: {cls_data['cluster'].nunique()}")

In [ ]:
def run_clustering_pipeline(all_embeddings_dict, min_freq_threshold=3, n_clusters=400):
    print(f"Параметры:")
    print(f"  Порог частоты СК: >= {min_freq_threshold}")
    print(f"  Количество кластеров: {n_clusters}")

    X, true_labels, tokens_info, indices, class_counts = prepare_all_tokens_with_threshold(
        all_embeddings_dict, min_freq_threshold
    )
    print("\nАнализ распределения частот...")
    freq_df, _ = analyze_frequency_distribution(all_embeddings_dict)
    kmeans, cluster_labels, metrics, label_encoder = perform_kmeans_clustering(
        X, true_labels, n_clusters
    )
    df_clusters, cluster_details = analyze_clusters_content(
        tokens_info, cluster_labels, true_labels
    )
    X_tsne = visualize_clusters_2d(X, cluster_labels, true_labels)

    if len(np.unique(true_labels)) >= 10:
        visualize_confusion_matrix_interactive(cluster_labels, true_labels)

    print(f"\nДанные:")
    print(f"  Токенов: {len(X)}")
    print(f"  Семантических классов: {len(np.unique(true_labels))}")
    print(f"\nКачество кластеризации:")
    print(f"  ARI: {metrics['ari']:.4f}")
    print(f"  V-measure: {metrics['v_measure']:.4f}")
    print(f"  Homogeneity: {metrics['homogeneity']:.4f}")
    print(f"  Completeness: {metrics['completeness']:.4f}")

    # if metrics['silhouette'] is not None:
    #     print(f"  Silhouette: {metrics['silhouette']:.4f}")

    results = {
        'X': X,
        'true_labels': true_labels,
        'cluster_labels': cluster_labels,
        'tokens_info': tokens_info,
        'kmeans': kmeans,
        'metrics': metrics,
        'df_clusters': df_clusters,
        'cluster_details': cluster_details,
        'X_tsne': X_tsne,
        'label_encoder': label_encoder,
        'params': {
            'min_freq_threshold': min_freq_threshold,
            'n_clusters': n_clusters
        }
    }
    return results

In [ ]:
def run_comparison_experiment(all_embeddings_dict, thresholds=[1, 2, 3, 5], n_clusters=400):
    all_results = {}
    comparison_data = []

    for threshold in thresholds:
        print(f"ПОРОГ: >= {threshold}")

        try:
            results = run_clustering_pipeline(
                all_embeddings_dict,
                min_freq_threshold=threshold,
                n_clusters=n_clusters
            )

            all_results[threshold] = results

            comparison_data.append({
                'threshold': threshold,
                'tokens': len(results['X']),
                'classes': len(np.unique(results['true_labels'])),
                'ari': results['metrics']['ari'],
                'v_measure': results['metrics']['v_measure'],
                'homogeneity': results['metrics']['homogeneity'],
                'completeness': results['metrics']['completeness'],
                'nmi': results['metrics']['nmi']
            })

        except Exception as e:
            print(f"Ошибка при пороге {threshold}: {e}")
            continue

    if comparison_data:
        df_comparison = pd.DataFrame(comparison_data)
        print(df_comparison.to_string(index=False))

        fig, axes = plt.subplots(2, 2, figsize=(15, 10))

        # 1. Количество токенов и классов
        axes[0, 0].plot(df_comparison['threshold'], df_comparison['tokens'], 'bo-', label='Токены')
        axes[0, 0].set_xlabel('Порог')
        axes[0, 0].set_ylabel('Токены', color='blue')
        axes[0, 0].tick_params(axis='y', labelcolor='blue')
        axes[0, 0].grid(True, alpha=0.3)

        ax2 = axes[0, 0].twinx()
        ax2.plot(df_comparison['threshold'], df_comparison['classes'], 'ro-', label='Классы')
        ax2.set_ylabel('Классы', color='red')
        ax2.tick_params(axis='y', labelcolor='red')

        lines1, labels1 = axes[0, 0].get_legend_handles_labels()
        lines2, labels2 = ax2.get_legend_handles_labels()
        axes[0, 0].legend(lines1 + lines2, labels1 + labels2, loc='upper right')
        axes[0, 0].set_title('Токены и классы по порогам')

        # 2. ARI и V-measure
        axes[0, 1].plot(df_comparison['threshold'], df_comparison['ari'], 'go-', label='ARI', marker='o')
        axes[0, 1].plot(df_comparison['threshold'], df_comparison['v_measure'], 'mo-', label='V-measure', marker='s')
        axes[0, 1].set_xlabel('Порог')
        axes[0, 1].set_ylabel('Метрика')
        axes[0, 1].set_title('ARI и V-measure по порогам')
        axes[0, 1].legend()
        axes[0, 1].grid(True, alpha=0.3)

        # 3. Homogeneity и Completeness
        axes[1, 0].plot(df_comparison['threshold'], df_comparison['homogeneity'], 'co-', label='Homogeneity', marker='^')
        axes[1, 0].plot(df_comparison['threshold'], df_comparison['completeness'], 'yo-', label='Completeness', marker='v')
        axes[1, 0].set_xlabel('Порог')
        axes[1, 0].set_ylabel('Метрика')
        axes[1, 0].set_title('Homogeneity и Completeness по порогам')
        axes[1, 0].legend()
        axes[1, 0].grid(True, alpha=0.3)

        # 4. NMI
        axes[1, 1].plot(df_comparison['threshold'], df_comparison['nmi'], 'ko-', label='NMI')
        axes[1, 1].set_xlabel('Порог')
        axes[1, 1].set_ylabel('NMI')
        axes[1, 1].set_title('NMI по порогам')
        axes[1, 1].legend()
        axes[1, 1].grid(True, alpha=0.3)

        plt.suptitle(f'Сравнение кластеризации по разным порогам частоты (k={n_clusters})', fontsize=16)
        plt.tight_layout()
        plt.show()

        # Находим лучший порог
        best_ari_idx = df_comparison['ari'].idxmax()
        best_v_idx = df_comparison['v_measure'].idxmax()

        print(f"\nЛучшие результаты:")
        print(f"  По ARI: порог {df_comparison.loc[best_ari_idx, 'threshold']} (ARI={df_comparison.loc[best_ari_idx, 'ari']:.4f})")
        print(f"  По V-measure: порог {df_comparison.loc[best_v_idx, 'threshold']} (V={df_comparison.loc[best_v_idx, 'v_measure']:.4f})")

    return all_results, df_comparison if 'df_comparison' in locals() else None

In [ ]:
def save_clustering_results(results, filename_prefix='clustering'):
    import pickle
    import json

    with open(f'{filename_prefix}_results.pkl', 'wb') as f:
        pickle.dump(results, f)

    if 'df_clusters' in results:
        results['df_clusters'].to_csv(f'{filename_prefix}_data.csv', index=False)

    metrics = results.get('metrics', {})
    params = results.get('params', {})

    summary = {
        'parameters': params,
        'data_info': {
            'n_tokens': len(results['X']) if 'X' in results else 0,
            'n_classes': len(np.unique(results['true_labels'])) if 'true_labels' in results else 0,
            'n_clusters_found': len(np.unique(results['cluster_labels'])) if 'cluster_labels' in results else 0
        },
        'metrics': {k: (float(v) if v is not None else None) for k, v in metrics.items()}
    }

    with open(f'{filename_prefix}_summary.json', 'w', encoding='utf-8') as f:
        json.dump(summary, f, indent=2, ensure_ascii=False)

    print(f"\nРезультаты сохранены:")
    print(f"  {filename_prefix}_results.pkl - полные результаты")
    print(f"  {filename_prefix}_data.csv - таблица с данными")
    print(f"  {filename_prefix}_summary.json - сводка с метриками")

### Getting cl

In [ ]:
results = run_clustering_pipeline(
    all_embeddings,
    min_freq_threshold=1,
    n_clusters=400
)

save_clustering_results(results, 'clustering_single')